In [117]:
import pandas as pd
import glob
import os


In [118]:
df = pd.read_excel("medicine.xlsx")

print(df.shape)
df.head()


/Users/ayesha/Desktop/project_data/.venv/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


(38537, 31)


,Namn,Styrka,Form,ATC-kod,Verksamt ämne (förenklat),Verksamt ämne,Registrerings-status,Godkännande-datum,Avregistrerings-datum,"Tillfälligt återkallad, datum",...,Ombud,MT-nummer,NPL-id,Godkännande-procedur,EUMA-nummer,Produktkategori,Maskinell dos.disp,Tidigare läkemedelsnamn,Utbytbarhet,Dispens-beslut
0,A-Flydin Duroplex,NaN,Tablett,NaN,NaN,Arnica montana (arnika) torkad rot\nCrataegus ...,Registrerad,2011-03-04,NaT,NaT,...,NaN,27599,20221128007052,Nationella proceduren,NaN,Homeopatiskt läkemedel,Nej,NaN,Nej,Nej
1,Abacavir/Lamivudine Accord,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavir\nlamivudin,Godkänd,2016-03-31,NaT,NaT,...,Accord Healthcare AB,52918,20150416000034,Decentrala proceduren,NaN,Läkemedel,Nej,Iviverz,Ja,Nej
2,Abacavir/Lamivudine Glenmark,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirhydrokloridmonohydrat\nlamivudin,Godkänd,2018-05-22,NaT,NaT,...,Glenmark Pharmaceuticals Nordic AB,57475,20180129000014,Proceduren för ömsesidigt erkännande,NaN,Läkemedel,Nej,NaN,Ja,Nej
3,Abacavir/Lamivudine Medical Valley,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirsulfat\nlamivudin,Godkänd,2019-03-25,NaT,NaT,...,NaN,57373,20171222000018,Decentrala proceduren,NaN,Läkemedel,Nej,NaN,Ja,Nej
4,Abacavir/Lamivudine Sandoz,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirhydrokloridmonohydrat\nlamivudin,Godkänd,2016-08-11,NaT,NaT,...,NaN,52395,20141230000119,Proceduren för ömsesidigt erkännande,NaN,Läkemedel,Nej,NaN,Ja,Nej


In [103]:
columns_needed = [
    "Namn",
    "Styrka",
    "Form",
    "ATC-kod",
    "Verksamt ämne (förenklat)",
    "Verksamt ämne",
    "Registrerings-status",
    "Försäljningsstatus",
    "H/V",
    "Receptstatus",
    "NPL-id"
]

medicines = df[columns_needed].copy()

print(medicines.shape)
medicines.head()

(38537, 11)


,Namn,Styrka,Form,ATC-kod,Verksamt ämne (förenklat),Verksamt ämne,Registrerings-status,Försäljningsstatus,H/V,Receptstatus,NPL-id
0,A-Flydin Duroplex,NaN,Tablett,NaN,NaN,Arnica montana (arnika) torkad rot\nCrataegus ...,Registrerad,Ej aktuellt,HUM,Receptfritt,20221128007052
1,Abacavir/Lamivudine Accord,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavir\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20150416000034
2,Abacavir/Lamivudine Glenmark,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirhydrokloridmonohydrat\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20180129000014
3,Abacavir/Lamivudine Medical Valley,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirsulfat\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20171222000018
4,Abacavir/Lamivudine Sandoz,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirhydrokloridmonohydrat\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20141230000119


In [104]:
medicines = medicines[
    (medicines["H/V"] == "HUM") &
    (medicines["Registrerings-status"] == "Godkänd")
].copy()

print("Rows after filtering:", len(medicines))

Rows after filtering: 15190


In [105]:
medicines = medicines.rename(columns={
    "Namn": "product_name",
    "Styrka": "strength",
    "Form": "form",
    "ATC-kod": "atc_code",
    "Verksamt ämne (förenklat)": "active_ingredient_simple",
    "Verksamt ämne": "active_ingredient",
    "Registrerings-status": "registration_status",
    "Försäljningsstatus": "sales_status",
    "H/V": "human_vet",
    "Receptstatus": "prescription_status",
    "NPL-id": "npl_id"
})

print(medicines.columns.tolist())

['product_name', 'strength', 'form', 'atc_code', 'active_ingredient_simple', 'active_ingredient', 'registration_status', 'sales_status', 'human_vet', 'prescription_status', 'npl_id']


In [106]:
print(medicines.isnull().sum())

product_name                  0
strength                    432
form                          0
atc_code                      0
active_ingredient_simple      2
active_ingredient             2
registration_status           0
sales_status                  0
human_vet                     0
prescription_status           0
npl_id                        0
dtype: int64


In [107]:
missing = pd.DataFrame({
    "missing_count": medicines.isnull().sum(),
    "missing_percent": (
        medicines.isnull().sum() / len(medicines) * 100
    ).round(2)
})

missing

,missing_count,missing_percent
product_name,0,0.00
strength,432,2.84
form,0,0.00
atc_code,0,0.00
active_ingredient_simple,2,0.01
active_ingredient,2,0.01
registration_status,0,0.00
sales_status,0,0.00
human_vet,0,0.00
prescription_status,0,0.00


In [108]:
medicines.loc[
    medicines["strength"].isna(),
    [
        "product_name",
        "strength",
        "form",
        "atc_code",
        "active_ingredient",
        "npl_id"
    ]
].head(30)

,product_name,strength,form,atc_code,active_ingredient,npl_id
19,Abdomin Citrus,NaN,Pulver till oral lösning i dospåse,A06AD65,kaliumklorid\nmakrogol\nnatriumklorid\nnatrium...,20240318000178
20,Abdomin Junior Neutral,NaN,Pulver till oral lösning i dospåse,A06AD65,kaliumklorid\nmakrogol\nnatriumklorid\nnatrium...,20240318000185
21,Abdomin Neutral,NaN,Pulver till oral lösning i dospåse,A06AD65,kaliumklorid\nmakrogol\nnatriumklorid\nnatrium...,20240318000161
22,Abecma,NaN,"Infusionsvätska, dispersion",L01XL07,idekabtagen-vikleucel,20200430000074
66,Abrysvo,NaN,"Pulver och vätska till injektionsvätska, lösning",J07BX05,"respiratoriskt syncytialt virus, subgrupp A, p...",20221223000033
67,Abrysvo,NaN,"Pulver och vätska till injektionsvätska, lösni...",J07BX05,"respiratoriskt syncytialt virus, subgrupp A, p...",20250616000039
160,Act-HIB,NaN,"Pulver och vätska till injektionsvätska, lösning",J07AG01,Haemophilus influenzae typ B polysackarid,19920612000022
168,Actilyse,NaN,Pulver och vätska till injektions-/infusionsvä...,B01AD02,alteplas,19900504000200
169,Actilyse,NaN,Pulver och vätska till injektions-/infusionsvä...,B01AD02,alteplas,20251105000073
170,Actilyse,NaN,Pulver och vätska till injektions-/infusionsvä...,B01AD02,alteplas,20130904000050


In [109]:
medicines.loc[
    medicines["active_ingredient"].isna(),
    [
        "product_name",
        "strength",
        "form",
        "atc_code",
        "active_ingredient_simple",
        "active_ingredient",
        "npl_id"
    ]
]

,product_name,strength,form,atc_code,active_ingredient_simple,active_ingredient,npl_id
498,ALK-Diluent,NaN,Spädningsvätska för parenteral användning,V07AB,NaN,NaN,19901102000524
14589,Soluprick Negativ Kontroll,NaN,Lösning för pricktest,V04CL,NaN,NaN,19901207000351


In [110]:
text_columns = [
    "product_name",
    "strength",
    "form",
    "atc_code",
    "active_ingredient_simple",
    "active_ingredient",
    "registration_status",
    "sales_status",
    "human_vet",
    "prescription_status"
]

for col in text_columns:
    medicines[col] = medicines[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

In [111]:
print("Total rows:", len(medicines))
print("Duplicate complete rows:", medicines.duplicated().sum())
print("Duplicate NPL IDs:", medicines["npl_id"].duplicated().sum())

Total rows: 15190
Duplicate complete rows: 0
Duplicate NPL IDs: 0


In [112]:
print("Unique product names:", medicines["product_name"].nunique())
print("Unique NPL IDs:", medicines["npl_id"].nunique())
print("Missing NPL IDs:", medicines["npl_id"].isna().sum())

Unique product names: 6768
Unique NPL IDs: 15190
Missing NPL IDs: 0


In [113]:
medicines = medicines.reset_index(drop=True)

medicines.head()

,product_name,strength,form,atc_code,active_ingredient_simple,active_ingredient,registration_status,sales_status,human_vet,prescription_status,npl_id
0,Abacavir/Lamivudine Accord,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavir\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20150416000034
1,Abacavir/Lamivudine Glenmark,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirhydrokloridmonohydrat\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20180129000014
2,Abacavir/Lamivudine Medical Valley,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirsulfat\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20171222000018
3,Abacavir/Lamivudine Sandoz,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirhydrokloridmonohydrat\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20141230000119
4,Abacavir/Lamivudine STADA,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavir\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20151021000044


In [119]:
medicines.to_csv(
    "medicines_master.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", len(medicines), "rows")

Saved: 15190 rows


In [120]:
check_df = pd.read_csv("medicines_master.csv")

print("Shape:", check_df.shape)
print("Unique NPL IDs:", check_df["npl_id"].nunique())
print("Missing NPL IDs:", check_df["npl_id"].isna().sum())

check_df.head()

Shape: (15190, 11)
Unique NPL IDs: 15190
Missing NPL IDs: 0


,product_name,strength,form,atc_code,active_ingredient_simple,active_ingredient,registration_status,sales_status,human_vet,prescription_status,npl_id
0,Abacavir/Lamivudine Accord,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavir\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20150416000034
1,Abacavir/Lamivudine Glenmark,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirhydrokloridmonohydrat\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20180129000014
2,Abacavir/Lamivudine Medical Valley,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirsulfat\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20171222000018
3,Abacavir/Lamivudine Sandoz,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirhydrokloridmonohydrat\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20141230000119
4,Abacavir/Lamivudine STADA,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavir\nlamivudin,Godkänd,Finns till försäljning,HUM,Receptbelagt,20151021000044


In [122]:

df = pd.read_csv(
    "data_matching.csv",
    sep=";",
    encoding="utf-8-sig",
    decimal=","
)

filtered = df[
    (df["Mått"] == 2) &
    (df["År"] == 2025) &
    (df["Region"] == 0) &
    (df["Kön"] == 3) &
    (df["Ålder"] == 99)
].copy()

print("Rows after national filter:", len(filtered))


filtered["ATC-kod"] = filtered["ATC-kod"].astype(str).str.strip()

specific_drugs = filtered[
    filtered["ATC-kod"].str.match(r"^[A-Z]\d{2}[A-Z]{2}\d{2}$", na=False)
].copy()

print("Specific ATC substances:", len(specific_drugs))


atc = pd.read_csv(
    "data_matching_atc.csv",
    sep=";",
    encoding="utf-8-sig"
)

atc = atc.rename(columns={
    "ATC": "ATC-kod",
    "Text": "medicine_name"
})


specific_drugs = specific_drugs.merge(
    atc,
    on="ATC-kod",
    how="left"
)

specific_drugs = specific_drugs.sort_values(
    "Värde",
    ascending=False
)

top500 = specific_drugs.head(500).copy()

top500 = top500[
    ["ATC-kod", "medicine_name", "Värde"]
].rename(columns={
    "ATC-kod": "atc_code",
    "Värde": "patients_per_1000"
})


top500.to_csv(
    "top_500_common_medicines_sweden_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nTop 500 created successfully!")
display(top500.head(20))

Rows after national filter: 3049
Specific ATC substances: 2043

Top 500 created successfully!


,atc_code,medicine_name,patients_per_1000
1483,N02BE01,Paracetamol,124.91
529,C10AA05,Atorvastatin,80.71
482,C08CA01,Amlodipin,71.04
22,A02BC01,Omeprazol,62.89
1877,R06AX27,Desloratadin,55.17
856,J01CE02,Fenoximetylpenicillin,54.99
471,C07AB02,Metoprolol,54.43
511,C09CA06,Kandesartan,50.69
91,A06AD65,"Makrogol, kombinationer",49.39
274,B01AC06,Acetylsalicylsyra,49.06


In [123]:
top500 = top500.rename(columns={
    "medicine_name": "active_ingredient"
})

In [124]:
top500.to_csv(
    "top_500_common_medicines_sweden_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

display(top500.head(10))

,atc_code,active_ingredient,patients_per_1000
1483,N02BE01,Paracetamol,124.91
529,C10AA05,Atorvastatin,80.71
482,C08CA01,Amlodipin,71.04
22,A02BC01,Omeprazol,62.89
1877,R06AX27,Desloratadin,55.17
856,J01CE02,Fenoximetylpenicillin,54.99
471,C07AB02,Metoprolol,54.43
511,C09CA06,Kandesartan,50.69
91,A06AD65,"Makrogol, kombinationer",49.39
274,B01AC06,Acetylsalicylsyra,49.06


In [125]:

fass = pd.read_csv(
    "fass_collection.csv",
    encoding="utf-8-sig"
)

print("Shape:", fass.shape)
print("\nColumns:")
print(fass.columns.tolist())

display(fass.head(10))

Shape: (14900, 17)

Columns:
['product_name', 'strength', 'form', 'atc_code', 'active_ingredient_simple', 'active_ingredient', 'prescription_status', 'sales_status', 'npl_id', 'source_url', 'fass_status', 'dosage', 'contraindications', 'warnings', 'interactions', 'pregnancy_breastfeeding', 'side_effects']


,product_name,strength,form,atc_code,active_ingredient_simple,active_ingredient,prescription_status,sales_status,npl_id,source_url,fass_status,dosage,contraindications,warnings,interactions,pregnancy_breastfeeding,side_effects
0,Abacavir/Lamivudine Accord,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavir\nlamivudin,Receptbelagt,Finns till försäljning,20150416000034,https://fass.se/health/product/20150416000034/...,smpc_available,4.2 Dosering och administreringssätt Behandlin...,4.3 Kontraindikationer Överkänslighet mot den ...,4.4 Varningar och försiktighet Varningar och f...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar Sammanfattning av säkerhetspr...
1,Abacavir/Lamivudine Glenmark,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirhydrokloridmonohydrat\nlamivudin,Receptbelagt,Finns till försäljning,20180129000014,https://fass.se/health/product/20180129000014/...,smpc_available,4.2 Dosering och administreringssätt Behandlin...,4.3 Kontraindikationer Överkänslighet mot de a...,4.4 Varningar och försiktighet Varningar och f...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar Sammanfattning av säkerhetspr...
2,Abacavir/Lamivudine Medical Valley,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirsulfat\nlamivudin,Receptbelagt,Finns till försäljning,20171222000018,https://fass.se/health/product/20171222000018/...,smpc_available,4.2 Dosering och administreringssätt Behandlin...,4.3 Kontraindikationer Överkänslighet mot den ...,4.4 Varningar och försiktighet Varningar och f...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar Sammanfattning av säkerhetspr...
3,Abacavir/Lamivudine Sandoz,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirhydrokloridmonohydrat\nlamivudin,Receptbelagt,Finns till försäljning,20141230000119,https://fass.se/health/product/20141230000119/...,smpc_available,4.2 Dosering och administreringssätt Behandlin...,4.3 Kontraindikationer Överkänslighet mot de a...,4.4 Varningar och försiktighet Varningar och f...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar Sammanfattning av säkerhetspr...
4,Abacavir/Lamivudine STADA,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavir\nlamivudin,Receptbelagt,Finns till försäljning,20151021000044,https://fass.se/health/product/20151021000044/...,smpc_available,4.2 Dosering och administreringssätt Behandlin...,4.3 Kontraindikationer Överkänslighet mot de a...,4.4 Varningar och försiktighet Varningar och f...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar Sammanfattning av säkerhetspr...
5,Abakavir/Lamivudin Viatris,600 mg/300 mg,Filmdragerad tablett,J05AR02,Abakavir + lamivudin,abakavirsulfat\nlamivudin,Receptbelagt,Finns till försäljning,20151229000068,https://fass.se/health/product/20151229000068/...,smpc_available,4.2 Dosering och administreringssätt Behandlin...,4.3 Kontraindikationer Överkänslighet mot de a...,4.4 Varningar och försiktighet Varningar och f...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar Sammanfattning av säkerhetspr...
6,ABASAGLAR,100 enheter/ml,"Injektionsvätska, lösning i cylinderampull",A10AE04,Insulin glargin,"insulin, glargin",Receptbelagt,Finns till försäljning,20130705000068,https://fass.se/health/product/20130705000068/...,smpc_available,4.2 Dosering och administreringssätt Dosering ...,4.3 Kontraindikationer Överkänslighet mot den ...,4.4 Varningar och försiktighet Spårbarhet För ...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar Summering av säkerhetspr

In [126]:

top500 = pd.read_csv(
    "top_500_common_medicines_sweden_2025.csv",
    encoding="utf-8-sig"
)

fass = pd.read_csv(
    "fass_collection.csv",
    encoding="utf-8-sig",
    dtype={"npl_id": str}
)

print("Top 500:", len(top500))
print("FASS products:", len(fass))

Top 500: 500
FASS products: 14900


In [127]:
top500["atc_code"] = (
    top500["atc_code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

fass["atc_code"] = (
    fass["atc_code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [128]:
matched = top500.merge(
    fass,
    on="atc_code",
    how="left",
    suffixes=("_socialstyrelsen", "_fass")
)

print("Rows after merge:", len(matched))

display(
    matched[
        [
            "atc_code",
            "active_ingredient_socialstyrelsen",
            "patients_per_1000",
            "product_name",
            "strength",
            "form",
            "fass_status"
        ]
    ].head(30)
)

Rows after merge: 8703


,atc_code,active_ingredient_socialstyrelsen,patients_per_1000,product_name,strength,form,fass_status
0,N02BE01,Paracetamol,124.91,Alvedon,125 mg,Suppositorium,smpc_available
1,N02BE01,Paracetamol,124.91,Alvedon,24 mg/ml,Oral lösning,smpc_available
2,N02BE01,Paracetamol,124.91,Alvedon,24 mg/ml,Oral suspension,smpc_available
3,N02BE01,Paracetamol,124.91,Alvedon,250 mg,Munsönderfallande tablett,smpc_available
4,N02BE01,Paracetamol,124.91,Alvedon,250 mg,Suppositorium,smpc_available
5,N02BE01,Paracetamol,124.91,Alvedon,500 mg,Brustablett,smpc_available
6,N02BE01,Paracetamol,124.91,Alvedon,500 mg,Filmdragerad tablett,smpc_available
7,N02BE01,Paracetamol,124.91,Alvedon,500 mg,Munsönderfallande tablett,smpc_available
8,N02BE01,Paracetamol,124.91,Alvedon,500 mg,Pulver till oral lösning i dospåse,limited_information
9,N02BE01,Paracetamol,124.91,Alvedon,500 mg,Suppositorium,smpc_available


In [129]:
coverage = matched.groupby("atc_code").agg(
    socialstyrelsen_name=("active_ingredient_socialstyrelsen", "first"),
    patients_per_1000=("patients_per_1000", "first"),
    fass_products=("product_name", "count"),
    smpc_products=("fass_status", lambda x: (x == "smpc_available").sum())
).reset_index()

print("Top ATCs:", len(top500))

print(
    "Matched with at least one FASS product:",
    (coverage["fass_products"] > 0).sum()
)

print(
    "Matched with at least one full SmPC:",
    (coverage["smpc_products"] > 0).sum()
)

print(
    "No FASS product:",
    (coverage["fass_products"] == 0).sum()
)

print(
    "FASS product exists but no full SmPC:",
    (
        (coverage["fass_products"] > 0) &
        (coverage["smpc_products"] == 0)
    ).sum()
)

Top ATCs: 500
Matched with at least one FASS product: 497
Matched with at least one full SmPC: 492
No FASS product: 3
FASS product exists but no full SmPC: 5


In [130]:
missing = coverage[
    coverage["fass_products"] == 0
].sort_values(
    "patients_per_1000",
    ascending=False
)

display(missing.head(30))

,atc_code,socialstyrelsen_name,patients_per_1000,fass_products,smpc_products
172,D08AX06,Kaliumpermanganat,0.63,0,0
487,S01XA03,"Natriumklorid, hyperton",0.60,0,0
16,A06AA01,Flytande paraffin,0.31,0,0


In [131]:
no_smpc = coverage[
    (coverage["fass_products"] > 0) &
    (coverage["smpc_products"] == 0)
].sort_values(
    "patients_per_1000",
    ascending=False
)

display(no_smpc.head(30))

,atc_code,socialstyrelsen_name,patients_per_1000,fass_products,smpc_products
82,B03AA01,Järn(II)glycinsulfat,5.01,6,0
169,D07CC01,Betametason och antibiotika,0.67,2,0
159,D06BB04,Podofyllotoxin,0.66,2,0
400,N07CA52,Cinnarizin kombinationer,0.38,5,0
225,G03HB01,Cyproteron och östrogen,0.33,1,0


In [132]:
# Keep only products with full SmPC
full_smpc = matched[
    matched["fass_status"] == "smpc_available"
].copy()

# Prefer products that are currently sold
full_smpc["currently_sold"] = (
    full_smpc["sales_status"]
    .astype(str)
    .str.contains("Finns till försäljning", case=False, na=False)
)

# Check how many ATCs have currently sold products
sold_check = full_smpc.groupby("atc_code")["currently_sold"].any()

print("ATCs with full SmPC:", full_smpc["atc_code"].nunique())
print("ATCs with currently sold full-SmPC product:", sold_check.sum())

ATCs with full SmPC: 492
ATCs with currently sold full-SmPC product: 490


In [133]:
# Sort so currently sold products come first
full_smpc = full_smpc.sort_values(
    by=["atc_code", "currently_sold"],
    ascending=[True, False]
)

# Select one representative FASS product for each ATC
selected = (
    full_smpc
    .drop_duplicates(subset="atc_code", keep="first")
    .copy()
)

print("Selected medicines:", len(selected))

display(
    selected[
        [
            "atc_code",
            "active_ingredient_socialstyrelsen",
            "patients_per_1000",
            "product_name",
            "strength",
            "form",
            "prescription_status",
            "sales_status",
            "fass_status"
        ]
    ].head(20)
)

Selected medicines: 492


,atc_code,active_ingredient_socialstyrelsen,patients_per_1000,product_name,strength,form,prescription_status,sales_status,fass_status
928,A01AA01,Natriumfluorid,31.35,Dentan,"0,05 %",Munsköljvätska,Receptfritt,Finns till försäljning,smpc_available
4281,A01AA30,Kombinationer,4.24,Xerodent,"28,6 mg/0,25 mg",Sugtablett,Receptfritt,Finns till försäljning,smpc_available
7258,A01AB03,Klorhexidin,0.77,Corsodyl,1 %,Munhålegel,Receptfritt,Finns till försäljning,smpc_available
5216,A01AD02,Bensydamin,3.01,Andolex,"1,5 mg/ml",Munsköljvätska,Receptbelagt,Finns till försäljning,smpc_available
8614,A02AD01,"Aluminium-, kalcium- och magnesiumsalter",0.32,Abecol,"180 mg/43,7 mg",Tuggtablett,Vissa förpackningar receptbelagda,Finns till försäljning,smpc_available
169,A02BC01,Omeprazol,62.89,Losec,10 mg,Enterotablett,Vissa förpackningar receptbelagda,Finns till försäljning,smpc_available
2584,A02BC02,Pantoprazol,9.34,Pantoprazol Actavis,20 mg,Enterotablett,Receptbelagt,Finns till försäljning,smpc_available
6360,A02BC03,Lansoprazol,1.29,Lansoprazol Krka,15 mg,"Enterokapsel, hård",Vissa förpackningar receptbelagda,Finns till försäljning,smpc_available
1262,A02BC05,Esomeprazol,27.33,Esomeprazol Actavis,20 mg,Enterotablett,Vissa förpackningar receptbelagda,Finns till försäljning,smpc_available
5434,A02BX13,Alginsyra,2.37,Galieve Peppermint,NaN,Tuggtablett,Receptfritt,Finns till försäljning,smpc_available


In [135]:
final490 = selected[
    selected["sales_status"]
    .astype(str)
    .str.contains("Finns till försäljning", case=False, na=False)
].copy()

print("Final medicines:", len(final490))

display(
    final490[
        [
            "atc_code",
            "active_ingredient_socialstyrelsen",
            "patients_per_1000",
            "product_name",
            "strength",
            "form",
            "prescription_status",
            "sales_status",
            "fass_status"
        ]
    ].head(20)
)

Final medicines: 490


,atc_code,active_ingredient_socialstyrelsen,patients_per_1000,product_name,strength,form,prescription_status,sales_status,fass_status
928,A01AA01,Natriumfluorid,31.35,Dentan,"0,05 %",Munsköljvätska,Receptfritt,Finns till försäljning,smpc_available
4281,A01AA30,Kombinationer,4.24,Xerodent,"28,6 mg/0,25 mg",Sugtablett,Receptfritt,Finns till försäljning,smpc_available
7258,A01AB03,Klorhexidin,0.77,Corsodyl,1 %,Munhålegel,Receptfritt,Finns till försäljning,smpc_available
5216,A01AD02,Bensydamin,3.01,Andolex,"1,5 mg/ml",Munsköljvätska,Receptbelagt,Finns till försäljning,smpc_available
8614,A02AD01,"Aluminium-, kalcium- och magnesiumsalter",0.32,Abecol,"180 mg/43,7 mg",Tuggtablett,Vissa förpackningar receptbelagda,Finns till försäljning,smpc_available
169,A02BC01,Omeprazol,62.89,Losec,10 mg,Enterotablett,Vissa förpackningar receptbelagda,Finns till försäljning,smpc_available
2584,A02BC02,Pantoprazol,9.34,Pantoprazol Actavis,20 mg,Enterotablett,Receptbelagt,Finns till försäljning,smpc_available
6360,A02BC03,Lansoprazol,1.29,Lansoprazol Krka,15 mg,"Enterokapsel, hård",Vissa förpackningar receptbelagda,Finns till försäljning,smpc_available
1262,A02BC05,Esomeprazol,27.33,Esomeprazol Actavis,20 mg,Enterotablett,Vissa förpackningar receptbelagda,Finns till försäljning,smpc_available
5434,A02BX13,Alginsyra,2.37,Galieve Peppermint,NaN,Tuggtablett,Receptfritt,Finns till försäljning,smpc_available


In [136]:
final490.to_csv(
    "final_490_medicines.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: final_490_medicines.csv")

Saved: final_490_medicines.csv


In [137]:
print("Shape:", final490.shape)

print("\nColumns:")
print(final490.columns.tolist())

print("\nMissing values:")
print(final490.isnull().sum())

print("\nDuplicate rows:", final490.duplicated().sum())

print("\nUnique ATC codes:", final490["atc_code"].nunique())

Shape: (490, 20)

Columns:
['atc_code', 'active_ingredient_socialstyrelsen', 'patients_per_1000', 'product_name', 'strength', 'form', 'active_ingredient_simple', 'active_ingredient_fass', 'prescription_status', 'sales_status', 'npl_id', 'source_url', 'fass_status', 'dosage', 'contraindications', 'warnings', 'interactions', 'pregnancy_breastfeeding', 'side_effects', 'currently_sold']

Missing values:
atc_code                              0
active_ingredient_socialstyrelsen     0
patients_per_1000                     0
product_name                          0
strength                             19
form                                  0
active_ingredient_simple              0
active_ingredient_fass                0
prescription_status                   0
sales_status                          0
npl_id                                0
source_url                            0
fass_status                           0
dosage                                0
contraindications                    

In [138]:
missing_interactions = final490[
    final490["interactions"].isna()
][
    [
        "atc_code",
        "active_ingredient_socialstyrelsen",
        "product_name",
        "source_url"
    ]
]

print("Missing interactions:", len(missing_interactions))
display(missing_interactions)

Missing interactions: 43


,atc_code,active_ingredient_socialstyrelsen,product_name,source_url
4281,A01AA30,Kombinationer,Xerodent,https://fass.se/health/product/19990910000017/...
7258,A01AB03,Klorhexidin,Corsodyl,https://fass.se/health/product/19780609000028/...
8308,A06AD12,Laktitol,Importal Ex-Lax,https://fass.se/health/product/19930305000024/...
2925,A06AD15,Makrogol,Forlax,https://fass.se/health/product/19991203000011/...
8341,A06AG02,Bisakodyl,Toilax,https://fass.se/health/product/19700306000015/...
7248,A06AX05,Prukaloprid,Prucalopride Holsten,https://fass.se/health/product/20211019000057/...
1315,C03CA01,Furosemid,Furix,https://fass.se/health/product/19801003000053/...
5218,C05AA01,Hydrokortison,Xyloproct,https://fass.se/health/product/19690124000017/...
5320,C05AA04,Prednisolon,Scheriproct,https://fass.se/health/product/19650729000010/...
7345,C07AG01,Labetalol,Trandate,https://fass.se/health/product/19811218000021/...


In [139]:
missing_interactions = final490[
    final490["interactions"].isna()
][
    [
        "atc_code",
        "active_ingredient_socialstyrelsen",
        "product_name",
        "source_url"
    ]
]

print("Missing interactions:", len(missing_interactions))
display(missing_interactions)

Missing interactions: 43


,atc_code,active_ingredient_socialstyrelsen,product_name,source_url
4281,A01AA30,Kombinationer,Xerodent,https://fass.se/health/product/19990910000017/...
7258,A01AB03,Klorhexidin,Corsodyl,https://fass.se/health/product/19780609000028/...
8308,A06AD12,Laktitol,Importal Ex-Lax,https://fass.se/health/product/19930305000024/...
2925,A06AD15,Makrogol,Forlax,https://fass.se/health/product/19991203000011/...
8341,A06AG02,Bisakodyl,Toilax,https://fass.se/health/product/19700306000015/...
7248,A06AX05,Prukaloprid,Prucalopride Holsten,https://fass.se/health/product/20211019000057/...
1315,C03CA01,Furosemid,Furix,https://fass.se/health/product/19801003000053/...
5218,C05AA01,Hydrokortison,Xyloproct,https://fass.se/health/product/19690124000017/...
5320,C05AA04,Prednisolon,Scheriproct,https://fass.se/health/product/19650729000010/...
7345,C07AG01,Labetalol,Trandate,https://fass.se/health/product/19811218000021/...


In [140]:
missing_pregnancy = final490[
    final490["pregnancy_breastfeeding"].isna()
][
    [
        "atc_code",
        "active_ingredient_socialstyrelsen",
        "product_name",
        "source_url"
    ]
]

print("Missing pregnancy/breastfeeding:", len(missing_pregnancy))
display(missing_pregnancy)


Missing pregnancy/breastfeeding: 44


,atc_code,active_ingredient_socialstyrelsen,product_name,source_url
4281,A01AA30,Kombinationer,Xerodent,https://fass.se/health/product/19990910000017/...
7258,A01AB03,Klorhexidin,Corsodyl,https://fass.se/health/product/19780609000028/...
8308,A06AD12,Laktitol,Importal Ex-Lax,https://fass.se/health/product/19930305000024/...
2925,A06AD15,Makrogol,Forlax,https://fass.se/health/product/19991203000011/...
8341,A06AG02,Bisakodyl,Toilax,https://fass.se/health/product/19700306000015/...
7248,A06AX05,Prukaloprid,Prucalopride Holsten,https://fass.se/health/product/20211019000057/...
1315,C03CA01,Furosemid,Furix,https://fass.se/health/product/19801003000053/...
5218,C05AA01,Hydrokortison,Xyloproct,https://fass.se/health/product/19690124000017/...
5320,C05AA04,Prednisolon,Scheriproct,https://fass.se/health/product/19650729000010/...
7345,C07AG01,Labetalol,Trandate,https://fass.se/health/product/19811218000021/...


In [141]:
both_missing = final490[
    final490["interactions"].isna() &
    final490["pregnancy_breastfeeding"].isna()
]

print("Both missing:", len(both_missing))

display(
    both_missing[
        [
            "atc_code",
            "active_ingredient_socialstyrelsen",
            "product_name"
        ]
    ]
)

Both missing: 42


,atc_code,active_ingredient_socialstyrelsen,product_name
4281,A01AA30,Kombinationer,Xerodent
7258,A01AB03,Klorhexidin,Corsodyl
8308,A06AD12,Laktitol,Importal Ex-Lax
2925,A06AD15,Makrogol,Forlax
8341,A06AG02,Bisakodyl,Toilax
7248,A06AX05,Prukaloprid,Prucalopride Holsten
1315,C03CA01,Furosemid,Furix
5218,C05AA01,Hydrokortison,Xyloproct
5320,C05AA04,Prednisolon,Scheriproct
7345,C07AG01,Labetalol,Trandate


In [142]:
furix = final490[
    final490["product_name"].str.contains(
        "Furix",
        case=False,
        na=False
    )
]

print("Product:", furix["product_name"].iloc[0])
print("ATC:", furix["atc_code"].iloc[0])
print("URL:", furix["source_url"].iloc[0])

print("\nInteractions:")
print(furix["interactions"].iloc[0])

print("\nPregnancy:")
print(furix["pregnancy_breastfeeding"].iloc[0])

Product: Furix
ATC: C03CA01
URL: https://fass.se/health/product/19801003000053/smpc

Interactions:
nan

Pregnancy:
nan


In [143]:
required_fields = [
    "dosage",
    "contraindications",
    "warnings",
    "interactions",
    "pregnancy_breastfeeding",
    "side_effects"
]

final_complete = final490.dropna(
    subset=required_fields
).copy()

print("Final complete medicines:", len(final_complete))

final_complete.to_csv(
    "final_complete_medicines.csv",
    index=False,
    encoding="utf-8-sig"
)

Final complete medicines: 444


In [144]:

df = pd.read_csv(
    "final_complete_medicines.csv",
    encoding="utf-8-sig",
    dtype={"npl_id": str}
)

text_columns = [
    "dosage",
    "contraindications",
    "warnings",
    "interactions",
    "pregnancy_breastfeeding",
    "side_effects"
]

def clean_text(text):
    if pd.isna(text):
        return text

    text = str(text)

    # Remove excessive whitespace/newlines
    text = re.sub(r"\s+", " ", text)

    # Remove spaces at beginning/end
    return text.strip()

for col in text_columns:
    df[col] = df[col].apply(clean_text)

print("Rows:", len(df))
print("Unique ATC:", df["atc_code"].nunique())
print("\nMissing values:")
print(df[text_columns].isna().sum())

display(df.head(3))

Rows: 444
Unique ATC: 444

Missing values:
dosage                     0
contraindications          0
warnings                   0
interactions               0
pregnancy_breastfeeding    0
side_effects               0
dtype: int64


,atc_code,active_ingredient_socialstyrelsen,patients_per_1000,product_name,strength,form,active_ingredient_simple,active_ingredient_fass,prescription_status,sales_status,npl_id,source_url,fass_status,dosage,contraindications,warnings,interactions,pregnancy_breastfeeding,side_effects,currently_sold
0,A01AA01,Natriumfluorid,31.35,Dentan,"0,05 %",Munsköljvätska,Fluorid,natriumfluorid,Receptfritt,Finns till försäljning,19730615000055,https://fass.se/health/product/19730615000055/...,smpc_available,4.2 Dosering och administreringssätt Dosering ...,4.3 Kontraindikationer 5 FARMAKOLOGISKA EGENSK...,4.4 Varningar och försiktighet Dentan innehåll...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Inga ris...",4.8 Biverkningar Biverkningarna är klassificer...,True
1,A01AD02,Bensydamin,3.01,Andolex,"1,5 mg/ml",Munsköljvätska,Bensydamin,bensydaminhydroklorid,Receptbelagt,Finns till försäljning,19951025000040,https://fass.se/health/product/19951025000040/...,smpc_available,4.2 Dosering och administreringssätt Dosering ...,4.3 Kontraindikationer Överkänslighet mot det ...,4.4 Varningar och försiktighet Andolex innehål...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Fertilit...",4.8 Biverkningar Biverkningar anges nedan efte...,True
2,A02AD01,"Aluminium-, kalcium- och magnesiumsalter",0.32,Abecol,"180 mg/43,7 mg",Tuggtablett,Kalciumkarbonat + magnesiumhydroxid,kalciumkarbonat\nmagnesiumhydroxid,Vissa förpackningar receptbelagda,Finns till försäljning,20190619000059,https://fass.se/health/product/20190619000059/...,smpc_available,4.2 Dosering och administreringssätt 1-2 table...,4.3 Kontraindikationer Överkänslighet mot de a...,4.4 Varningar och försiktighet Långvarig använ...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar De rapporterade biverkningsfr...,True


In [145]:
df.to_csv(
    "medicines_444_cleaned.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: medicines_444_cleaned.csv")

Saved: medicines_444_cleaned.csv


In [ ]:
common444 = pd.read_csv(
    "medicines_444_cleaned.csv",
    encoding="utf-8-sig"
)


fass = pd.read_csv(
    "fass_collection.csv",
    encoding="utf-8-sig",
    dtype={"npl_id": str}
)

# Clean ATC codes
common444["atc_code"] = common444["atc_code"].astype(str).str.strip().str.upper()
fass["atc_code"] = fass["atc_code"].astype(str).str.strip().str.upper()

# 3. Get the 444 selected ATC codes
selected_atcs = set(common444["atc_code"])

# 4. Take ALL FASS products belonging to those 444 ATCs
all_variations = fass[
    fass["atc_code"].isin(selected_atcs)
].copy()

# 5. Keep only currently sold products
all_variations = all_variations[
    all_variations["sales_status"]
    .astype(str)
    .str.contains("Finns till försäljning", case=False, na=False)
].copy()

# 6. Keep products with full SmPC
all_variations = all_variations[
    all_variations["fass_status"] == "smpc_available"
].copy()

# 7. Remove exact duplicate NPL IDs only
all_variations = all_variations.drop_duplicates(
    subset=["npl_id"],
    keep="first"
)

print("Selected ATCs:", len(selected_atcs))
print("Unique ATCs found:", all_variations["atc_code"].nunique())
print("Total product variations:", len(all_variations))
print("Unique product names:", all_variations["product_name"].nunique())
print("Unique NPL IDs:", all_variations["npl_id"].nunique())

Selected ATCs: 444
Unique ATCs found: 444
Total product variations: 3990
Unique product names: 1924
Unique NPL IDs: 3990


In [66]:
alvedon = all_variations[
    all_variations["product_name"]
    .str.contains("Alvedon", case=False, na=False)
]

print("Alvedon variations:", len(alvedon))

display(
    alvedon[
        [
            "product_name",
            "strength",
            "form",
            "atc_code",
            "active_ingredient_simple",
            "npl_id"
        ]
    ].sort_values(["product_name", "strength", "form"])
)

Alvedon variations: 11


,product_name,strength,form,atc_code,active_ingredient_simple,npl_id
530,Alvedon,125 mg,Suppositorium,N02BE01,Paracetamol,19800125000064
532,Alvedon,24 mg/ml,Oral suspension,N02BE01,Paracetamol,20210406000014
533,Alvedon,250 mg,Munsönderfallande tablett,N02BE01,Paracetamol,20010601000075
534,Alvedon,250 mg,Suppositorium,N02BE01,Paracetamol,19920703000054
535,Alvedon,500 mg,Brustablett,N02BE01,Paracetamol,19750613000031
536,Alvedon,500 mg,Filmdragerad tablett,N02BE01,Paracetamol,19581115000027
537,Alvedon,500 mg,Munsönderfallande tablett,N02BE01,Paracetamol,20010601000082
539,Alvedon,500 mg,Suppositorium,N02BE01,Paracetamol,19780512000047
540,Alvedon,60 mg,Suppositorium,N02BE01,Paracetamol,19861212000084
543,Alvedon Novum,500 mg,Filmdragerad tablett,N02BE01,Paracetamol,20190527000028


In [67]:
print("Total product variations:", len(all_variations))
print("Unique ATC codes:", all_variations["atc_code"].nunique())
print("Unique product names:", all_variations["product_name"].nunique())
print("Unique NPL IDs:", all_variations["npl_id"].nunique())

print("\nMissing values:")
print(
    all_variations[
        [
            "product_name",
            "strength",
            "form",
            "active_ingredient_simple",
            "prescription_status",
            "dosage",
            "contraindications",
            "warnings",
            "interactions",
            "pregnancy_breastfeeding",
            "side_effects"
        ]
    ].isna().sum()
)

print("\nDuplicate NPL IDs:",
      all_variations["npl_id"].duplicated().sum())

Total product variations: 3990
Unique ATC codes: 444
Unique product names: 1924
Unique NPL IDs: 3990

Missing values:
product_name                  0
strength                     47
form                          0
active_ingredient_simple      0
prescription_status           0
dosage                        0
contraindications             2
warnings                      2
interactions                189
pregnancy_breastfeeding     191
side_effects                  0
dtype: int64

Duplicate NPL IDs: 0


In [68]:
all_variations.to_csv(
    "fass_444_atc_all_products.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved rows:", len(all_variations))
print("Saved as: fass_444_atc_all_products.csv")

Saved rows: 3990
Saved as: fass_444_atc_all_products.csv


In [69]:
df = pd.read_csv(
    "fass_444_atc_all_products.csv",
    encoding="utf-8-sig",
    dtype={"npl_id": str}
)

In [70]:
df

,product_name,strength,form,atc_code,active_ingredient_simple,active_ingredient,prescription_status,sales_status,npl_id,source_url,fass_status,dosage,contraindications,warnings,interactions,pregnancy_breastfeeding,side_effects
0,ABASAGLAR,100 enheter/ml,"Injektionsvätska, lösning i cylinderampull",A10AE04,Insulin glargin,"insulin, glargin",Receptbelagt,Finns till försäljning,20130705000068,https://fass.se/health/product/20130705000068/...,smpc_available,4.2 Dosering och administreringssätt Dosering ...,4.3 Kontraindikationer Överkänslighet mot den ...,4.4 Varningar och försiktighet Spårbarhet För ...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar Summering av säkerhetsprofile...
1,ABASAGLAR KwikPen,100 enheter/ml,"Injektionsvätska, lösning i förfylld injektion...",A10AE04,Insulin glargin,"insulin, glargin",Receptbelagt,Finns till försäljning,20130705000075,https://fass.se/health/product/20130705000075/...,smpc_available,4.2 Dosering och administreringssätt Dosering ...,4.3 Kontraindikationer Överkänslighet mot den ...,4.4 Varningar och försiktighet Spårbarhet För ...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar Summering av säkerhetsprofile...
2,Abboticin,1 g,"Pulver till infusionsvätska, lösning",J01FA01,Erytromycin,erytromycinlaktobionat,Receptbelagt,Finns till försäljning,19561113000014,https://fass.se/health/product/19561113000014/...,smpc_available,4.2 Dosering och administreringssätt Dosering ...,4.3 Kontraindikationer Överkänslighet mot den ...,4.4 Varningar och försiktighet Eftersom erytro...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar Den vanligast förekommande bi...
3,Abboticin,250 mg,Filmdragerad tablett,J01FA01,Erytromycin,erytromycinstearat,Receptbelagt,Finns till försäljning,20210621000011,https://fass.se/health/product/20210621000011/...,smpc_available,4.2 Dosering och administreringssätt Dosering ...,4.3 Kontraindikationer Överkänslighet mot eryt...,4.4 Varningar och försiktighet Erytromycin är ...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar De vanligaste rapporterade bi...
4,Abdomin Junior Neutral,NaN,Pulver till oral lösning i dospåse,A06AD65,Makrogol + elektrolyter,kaliumklorid\nmakrogol\nnatriumklorid\nnatrium...,Receptbelagt,Finns till försäljning,20240318000185,https://fass.se/health/product/20240318000185/...,smpc_available,4.2 Dosering och administreringssätt Dosering ...,4.3 Kontraindikationer Intestinal perforation ...,4.4 Varningar och försiktighet Vätskeinnehålle...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar De vanligast förekommande rea...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3985,Zopiklon Evolan,"7,5 mg",Filmdragerad tablett,N05CF01,Zopiklon,zopiklon,Receptbelagt,Finns till försäljning,20240409000025,https://fass.se/health/product/20240409000025/...,smpc_available,4.2 Dosering och administreringssätt Dosering ...,4.3 Kontraindikationer Överkänslighet mot den ...,4.4 Varningar och försiktighet Sömnsvårigheter...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar). Behandling med sömnmedel bö...
3986,Zopiklon Pilum,5 mg,Filmdragerad tablett,N05CF01,Zopiklon,zopiklon,Receptbelagt,Finns till försäljning,20110326000029,https://fass.se/health/product/20110326000029/...,smpc_available,4.2 Dosering och administreringssätt Dosering ...,4.3 Kontraindikationer Zopiklon Pilum är kontr...,4.4 Varningar och försiktighet Sömnsvårigheter...,4.5 Interaktioner med andra läkemedel och övri...,"4.6 Fertilitet, graviditet och amning Gravidit...",4.8 Biverkningar). Behandling med sömnmedel bö...
3987,Zopiklon Pilum,"7,5 mg",Filmdragerad tablett,N05CF01,

In [71]:
required_medical_fields = [
    "dosage",
    "contraindications",
    "warnings",
    "interactions",
    "pregnancy_breastfeeding",
    "side_effects"
]

complete_products = all_variations.dropna(
    subset=required_medical_fields
).copy()

print("Original FASS product records:", len(all_variations))
print("Complete FASS product records:", len(complete_products))
print("Incomplete records removed:",
      len(all_variations) - len(complete_products))

print("\nUnique ATC codes remaining:",
      complete_products["atc_code"].nunique())

print("Unique product names remaining:",
      complete_products["product_name"].nunique())

print("Unique NPL IDs remaining:",
      complete_products["npl_id"].nunique())

print("\nMissing strength among complete records:",
      complete_products["strength"].isna().sum())

Original FASS product records: 3990
Complete FASS product records: 3797
Incomplete records removed: 193

Unique ATC codes remaining: 444
Unique product names remaining: 1840
Unique NPL IDs remaining: 3797

Missing strength among complete records: 46


In [72]:
missing_strength = complete_products[
    complete_products["strength"].isna()
].copy()

print("Missing strength records:", len(missing_strength))

display(
    missing_strength[
        [
            "product_name",
            "form",
            "atc_code",
            "active_ingredient_simple",
            "npl_id"
        ]
    ]
)

Missing strength records: 46


,product_name,form,atc_code,active_ingredient_simple,npl_id
19,Abdomin Junior Neutral,Pulver till oral lösning i dospåse,A06AD65,Makrogol + elektrolyter,20240318000185
20,Abdomin Neutral,Pulver till oral lösning i dospåse,A06AD65,Makrogol + elektrolyter,20240318000161
977,Artelac,"Ögondroppar, lösning",S01XA20,Hypromellos,19990813000022
978,Artelac,"Ögondroppar, lösning i endosbehållare",S01XA20,Hypromellos,19990813000039
2370,Celluvisc,"Ögondroppar, lösning i endosbehållare",S01XA20,Karmellos,19970507000019
2577,CitraFleet,Pulver till oral lösning i dospåse,A06AB58,Natriumpikosulfat + magnesiumoxid + citronsyra,20070322000013
2686,Cocillana-Etyfin,Oral lösning,R05FA02,Etylmorfin + extrakt av cocillana + senega,19641222000010
2832,Creon 10000,"Enterokapsel, hård",A09AA02,Pankreaspulver,19851004000026
2837,Creon 25000,"Enterokapsel, hård",A09AA02,Pankreaspulver,19910705000053
2841,Creon 35000,"Enterokapsel, hård",A09AA02,Pankreaspulver,20170330000026


In [73]:
complete_products.to_csv(
    "fass_444_atc_complete_products.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved records:", len(complete_products))
print("Unique ATCs:", complete_products["atc_code"].nunique())
print("Unique product names:", complete_products["product_name"].nunique())
print("Missing strength:", complete_products["strength"].isna().sum())

Saved records: 3797
Unique ATCs: 444
Unique product names: 1840
Missing strength: 46


In [ ]:


df = pd.read_csv(
    "clean_data.csv",
    dtype={"npl_id": str}
)

print("Rows:", len(df))
print("Unique ATCs:", df["atc_code"].nunique())
print("Unique product names:", df["product_name"].nunique())

print("\nColumns:")
for col in df.columns:
    print("-", col)

df[
    [
        "product_name",
        "strength",
        "active_ingredient_simple",
        "atc_code"
    ]
].head(20)

Rows: 3797
Unique ATCs: 444
Unique product names: 1840

Columns:
- product_name
- strength
- form
- atc_code
- active_ingredient_simple
- active_ingredient
- prescription_status
- sales_status
- npl_id
- source_url
- fass_status
- dosage
- contraindications
- warnings
- interactions
- pregnancy_breastfeeding
- side_effects


,product_name,strength,active_ingredient_simple,atc_code
0,ABASAGLAR,100 enheter/ml,Insulin glargin,A10AE04
1,ABASAGLAR KwikPen,100 enheter/ml,Insulin glargin,A10AE04
2,Abboticin,1 g,Erytromycin,J01FA01
3,Abboticin,250 mg,Erytromycin,J01FA01
4,Abdomin Junior Neutral,NaN,Makrogol + elektrolyter,A06AD65
5,Abdomin Neutral,NaN,Makrogol + elektrolyter,A06AD65
6,Abecol,"180 mg/43,7 mg",Kalciumkarbonat + magnesiumhydroxid,A02AD01
7,Abilify,1 mg/ml,Aripiprazol,N05AX12
8,Abilify,10 mg,Aripiprazol,N05AX12
9,Abilify,10 mg,Aripiprazol,N05AX12


In [76]:
import re
import pandas as pd

def normalize_text(value):
    if pd.isna(value):
        return ""

    value = str(value).strip().lower()

    # Multiple spaces -> one space
    value = re.sub(r"\s+", " ", value)

    return value


df["brand_normalized"] = df["product_name"].apply(normalize_text)

df["generic_normalized"] = df["active_ingredient_simple"].apply(
    normalize_text
)

df["atc_normalized"] = (
    df["atc_code"]
    .astype(str)
    .str.strip()
    .str.upper()
)


def make_search_name(row):

    parts = [
        row["product_name"],
        row["strength"] if pd.notna(row["strength"]) else "",
        row["form"],
        row["active_ingredient_simple"],
        row["atc_code"]
    ]

    return " | ".join(
        str(x).strip()
        for x in parts
        if str(x).strip() and str(x).strip().lower() != "nan"
    )


df["search_name"] = df.apply(make_search_name, axis=1)

display(
    df[
        [
            "product_name",
            "brand_normalized",
            "strength",
            "active_ingredient_simple",
            "generic_normalized",
            "atc_normalized",
            "search_name"
        ]
    ].head(20)
)

,product_name,brand_normalized,strength,active_ingredient_simple,generic_normalized,atc_normalized,search_name
0,ABASAGLAR,abasaglar,100 enheter/ml,Insulin glargin,insulin glargin,A10AE04,"ABASAGLAR | 100 enheter/ml | Injektionsvätska,..."
1,ABASAGLAR KwikPen,abasaglar kwikpen,100 enheter/ml,Insulin glargin,insulin glargin,A10AE04,ABASAGLAR KwikPen | 100 enheter/ml | Injektion...
2,Abboticin,abboticin,1 g,Erytromycin,erytromycin,J01FA01,"Abboticin | 1 g | Pulver till infusionsvätska,..."
3,Abboticin,abboticin,250 mg,Erytromycin,erytromycin,J01FA01,Abboticin | 250 mg | Filmdragerad tablett | Er...
4,Abdomin Junior Neutral,abdomin junior neutral,NaN,Makrogol + elektrolyter,makrogol + elektrolyter,A06AD65,Abdomin Junior Neutral | Pulver till oral lösn...
5,Abdomin Neutral,abdomin neutral,NaN,Makrogol + elektrolyter,makrogol + elektrolyter,A06AD65,Abdomin Neutral | Pulver till oral lösning i d...
6,Abecol,abecol,"180 mg/43,7 mg",Kalciumkarbonat + magnesiumhydroxid,kalciumkarbonat + magnesiumhydroxid,A02AD01,"Abecol | 180 mg/43,7 mg | Tuggtablett | Kalciu..."
7,Abilify,abilify,1 mg/ml,Aripiprazol,aripiprazol,N05AX12,Abilify | 1 mg/ml | Oral lösning | Aripiprazol...
8,Abilify,abilify,10 mg,Aripiprazol,aripiprazol,N05AX12,Abilify | 10 mg | Munsönderfallande tablett | ...
9,Abilify,abilify,10 mg,Aripiprazol,aripiprazol,N05AX12,Abilify | 10 mg | Tablett | Aripiprazol | N05AX12


In [77]:
df.to_csv(
    "fass_444_normalized.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved: fass_444_normalized.csv")
print("Rows:", len(df))
print("Unique ATCs:", df["atc_normalized"].nunique())
print("Unique product names:", df["product_name"].nunique())

Saved: fass_444_normalized.csv
Rows: 3797
Unique ATCs: 444
Unique product names: 1840


In [82]:
import pandas as pd

df = pd.read_csv(
    "fass_444_normalized.csv",
    dtype={"npl_id": str}
)

# কয়েকটি interaction example দেখব
samples = df[
    [
        "product_name",
        "strength",
        "active_ingredient_simple",
        "atc_code",
        "interactions"
    ]
].dropna(subset=["interactions"]).head(10)

for _, row in samples.iterrows():
    print("=" * 80)
    print("PRODUCT:", row["product_name"])
    print("STRENGTH:", row["strength"])
    print("ACTIVE INGREDIENT:", row["active_ingredient_simple"])
    print("ATC:", row["atc_code"])
    print("\nINTERACTIONS:")
    print(row["interactions"])
    print()

PRODUCT: ABASAGLAR
STRENGTH: 100 enheter/ml
ACTIVE INGREDIENT: Insulin glargin
ATC: A10AE04

INTERACTIONS:
4.5 Interaktioner med andra läkemedel och övriga interaktioner Ett antal substanser påverkar glukosomsättningen och kan kräva dosanpassning av insulin glargin. Substanser som kan öka den blodsockersänkande effekten och öka benägenheten för hypoglykemi inkluderar orala antidiabetika, ACE-hämmare, disopyramid, fibrater, fluoxetin, monoaminoxidas (MAO) -hämmare, pentoxifyllin, propoxifen, salicylater, somastostatinanaloger och sulfonamidantibiotika. Substanser som kan minska den blodsockersänkande effekten inkluderar kortikosteroider, danazol, diazoxid, diuretika, glukagon, isoniazid, östrogener och progestogener, fentiazinderivat, somatropin, sympatomimetika (t ex epinefrin [adrenalin], salbutamol, terbutalin), tyroideahormoner, atypiska antipsykotiska läkemedel (t ex klozapin och olanzapin) och proteashämmare. Betablockerare, klonidin, litiumsalter och alkohol kan antingen förstärk

In [83]:


import pandas as pd

df = pd.read_csv(
    "fass_444_normalized.csv",
    encoding="utf-8-sig",
    dtype={"npl_id": str}
)

interaction_df = df[
    df["interactions"].notna() &
    (df["interactions"].astype(str).str.strip() != "")
].copy()

print("Rows with interaction text:", len(interaction_df))
print("Unique ATCs:", interaction_df["atc_code"].nunique())

for i, row in interaction_df.head(20).iterrows():
    print("=" * 100)
    print("PRODUCT:", row["product_name"])
    print("GENERIC:", row["active_ingredient_simple"])
    print("ATC:", row["atc_code"])
    print("SOURCE:", row["source_url"])
    print()
    print(row["interactions"])
    print()

Rows with interaction text: 3797
Unique ATCs: 444
PRODUCT: ABASAGLAR
GENERIC: Insulin glargin
ATC: A10AE04
SOURCE: https://fass.se/health/product/20130705000068/smpc

4.5 Interaktioner med andra läkemedel och övriga interaktioner Ett antal substanser påverkar glukosomsättningen och kan kräva dosanpassning av insulin glargin. Substanser som kan öka den blodsockersänkande effekten och öka benägenheten för hypoglykemi inkluderar orala antidiabetika, ACE-hämmare, disopyramid, fibrater, fluoxetin, monoaminoxidas (MAO) -hämmare, pentoxifyllin, propoxifen, salicylater, somastostatinanaloger och sulfonamidantibiotika. Substanser som kan minska den blodsockersänkande effekten inkluderar kortikosteroider, danazol, diazoxid, diuretika, glukagon, isoniazid, östrogener och progestogener, fentiazinderivat, somatropin, sympatomimetika (t ex epinefrin [adrenalin], salbutamol, terbutalin), tyroideahormoner, atypiska antipsykotiska läkemedel (t ex klozapin och olanzapin) och proteashämmare. Betablockera

In [85]:
import pandas as pd
import re

# Load your normalized dataset
df = pd.read_csv(
    "fass_444_normalized.csv",
    encoding="utf-8-sig",
    dtype={"npl_id": str}
)

# Keep rows that actually contain interaction text
interaction_df = df[
    df["interactions"].notna() &
    (df["interactions"].astype(str).str.strip() != "")
].copy()

def clean_interaction_text(value):
    if pd.isna(value):
        return ""

    text = str(value)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

interaction_df["interaction_text_clean"] = (
    interaction_df["interactions"]
    .apply(clean_interaction_text)
)

# Build interaction_records
interaction_records = interaction_df[
    [
        "product_name",
        "strength",
        "form",
        "active_ingredient_simple",
        "atc_code",
        "npl_id",
        "source_url",
        "interaction_text_clean"
    ]
].copy()

interaction_records = interaction_records.rename(
    columns={
        "active_ingredient_simple": "drug_a",
        "atc_code": "drug_a_atc",
        "interaction_text_clean": "interaction_text"
    }
)

print("interaction_records created:", len(interaction_records))
interaction_records.head()

interaction_records created: 3797


,product_name,strength,form,drug_a,drug_a_atc,npl_id,source_url,interaction_text
0,ABASAGLAR,100 enheter/ml,"Injektionsvätska, lösning i cylinderampull",Insulin glargin,A10AE04,20130705000068,https://fass.se/health/product/20130705000068/...,4.5 Interaktioner med andra läkemedel och övri...
1,ABASAGLAR KwikPen,100 enheter/ml,"Injektionsvätska, lösning i förfylld injektion...",Insulin glargin,A10AE04,20130705000075,https://fass.se/health/product/20130705000075/...,4.5 Interaktioner med andra läkemedel och övri...
2,Abboticin,1 g,"Pulver till infusionsvätska, lösning",Erytromycin,J01FA01,19561113000014,https://fass.se/health/product/19561113000014/...,4.5 Interaktioner med andra läkemedel och övri...
3,Abboticin,250 mg,Filmdragerad tablett,Erytromycin,J01FA01,20210621000011,https://fass.se/health/product/20210621000011/...,4.5 Interaktioner med andra läkemedel och övri...
4,Abdomin Junior Neutral,NaN,Pulver till oral lösning i dospåse,Makrogol + elektrolyter,A06AD65,20240318000185,https://fass.se/health/product/20240318000185/...,4.5 Interaktioner med andra läkemedel och övri...


In [89]:
import re
import pandas as pd

def extract_section_45(text):
    if pd.isna(text):
        return ""

    text = str(text).replace("\xa0", " ").strip()

    heading_pattern = re.compile(
        r"4\.5\s+Interaktioner med andra läkemedel och övriga interaktioner",
        flags=re.I
    )

    # Find ALL occurrences of the 4.5 heading
    matches = list(heading_pattern.finditer(text))

    if matches:
        # Important: use the LAST occurrence
        # Some scraped pages contain an earlier broken/duplicated 4.5 reference
        start = matches[-1].end()
        text = text[start:]

    # Stop when the next SmPC section begins
    next_section = re.search(
        r"\s+(?:4\.6|4\.7|4\.8|4\.9|5\.1)\s+",
        text
    )

    if next_section:
        text = text[:next_section.start()]

    text = re.sub(r"\s+", " ", text).strip()

    return text


interaction_records["section_45"] = (
    interaction_records["interaction_text"]
    .apply(extract_section_45)
)

In [91]:
# NEXT CELL
# Split section_45 into smaller interaction statements

import re

def split_interaction_statements(text):
    if not text:
        return []

    parts = re.split(
        r"(?<=[.!?])\s+(?=[A-ZÅÄÖ])",
        str(text)
    )

    return [
        p.strip()
        for p in parts
        if len(p.strip()) >= 20
    ]


interaction_records["interaction_statements"] = (
    interaction_records["section_45"]
    .apply(split_interaction_statements)
)

interaction_records[
    ["product_name", "drug_a", "interaction_statements"]
].head(10)

,product_name,drug_a,interaction_statements
0,ABASAGLAR,Insulin glargin,[Ett antal substanser påverkar glukosomsättnin...
1,ABASAGLAR KwikPen,Insulin glargin,[Ett antal substanser påverkar glukosomsättnin...
2,Abboticin,Erytromycin,[Effekt av Abboticin på andra läkemedel: Erytr...
3,Abboticin,Erytromycin,[Effekt av erytromycin på andra läkemedel Eryt...
4,Abdomin Junior Neutral,Makrogol + elektrolyter,[Perorala läkemedel i fast form som tas inom e...
5,Abdomin Neutral,Makrogol + elektrolyter,"[Makrogol ökar lösligheten av läkemedel, som ä..."
6,Abecol,Kalciumkarbonat + magnesiumhydroxid,[Estramustin : Antacida bildar svårlösliga sal...
7,Abilify,Aripiprazol,[På grund av att aripiprazol har en antagonist...
8,Abilify,Aripiprazol,[På grund av att aripiprazol har en antagonist...
9,Abilify,Aripiprazol,[På grund av att aripiprazol har en antagonist...


In [92]:
for _, row in interaction_records.head(5).iterrows():
    print("=" * 100)
    print("PRODUCT:", row["product_name"])
    print("DRUG A:", row["drug_a"])

    for i, s in enumerate(row["interaction_statements"][:10], start=1):
        print(f"{i}. {s}")

    print()

PRODUCT: ABASAGLAR
DRUG A: Insulin glargin
1. Ett antal substanser påverkar glukosomsättningen och kan kräva dosanpassning av insulin glargin.
2. Substanser som kan öka den blodsockersänkande effekten och öka benägenheten för hypoglykemi inkluderar orala antidiabetika, ACE-hämmare, disopyramid, fibrater, fluoxetin, monoaminoxidas (MAO) -hämmare, pentoxifyllin, propoxifen, salicylater, somastostatinanaloger och sulfonamidantibiotika.
3. Substanser som kan minska den blodsockersänkande effekten inkluderar kortikosteroider, danazol, diazoxid, diuretika, glukagon, isoniazid, östrogener och progestogener, fentiazinderivat, somatropin, sympatomimetika (t ex epinefrin [adrenalin], salbutamol, terbutalin), tyroideahormoner, atypiska antipsykotiska läkemedel (t ex klozapin och olanzapin) och proteashämmare.
4. Betablockerare, klonidin, litiumsalter och alkohol kan antingen förstärka eller minska insulinets blodsockersänkande effekt.
5. Pentamidin kan förorsaka hypoglykemi som ibland kan följas 

In [93]:
# NEXT CELL — build known drug dictionary

import re
import pandas as pd

drug_lookup = (
    df[["active_ingredient_simple", "atc_code"]]
    .dropna()
    .drop_duplicates()
    .copy()
)

drug_lookup["drug_name_norm"] = (
    drug_lookup["active_ingredient_simple"]
    .astype(str)
    .str.lower()
    .str.strip()
)

drug_name_to_atc = dict(
    zip(
        drug_lookup["drug_name_norm"],
        drug_lookup["atc_code"]
    )
)

known_drugs = sorted(
    drug_name_to_atc.keys(),
    key=len,
    reverse=True
)

print("Known drugs:", len(known_drugs))
print(known_drugs[:20])

Known drugs: 448
['alginsyra + natriumvätekarbonat + kalciumkarbonat + aluminiumoxid', 'makrogol + natriumsulfat + askorbinsyra + elektrolyter', 'beklometasondipropionat + formoterol + glykopyrronium', 'alginsyra + natriumvätekarbonat + kalciumkarbonat', 'alginsyra + natriumvätekarbonat + aluminiumoxid', 'natriumpikosulfat + magnesiumoxid + citronsyra', 'hydrokortison + oxitetracyklin + polymyxin b', 'flutikasonfuroat + umeklidinium + vilanterol', 'etylmorfin + extrakt av cocillana + senega', 'relugolix + estradiol + noretisteronacetat', 'insulin lispro + insulin lispro protamin', 'natriumcitrat + natriumlaurylsulfoacetat', 'insulin aspart + insulin aspart protamin', 'makrogol + natriumsulfat + elektrolyter', 'formoterol + glykopyrronium + budesonid', 'allergen, pollen från flera olika gräs', 'kandesartancilexetil + hydroklortiazid', 'beklometasondipropionat + formoterol', 'estradiol + medroxiprogesteronacetat', 'kodein + acetylsalicylsyra + koffein']


In [94]:
# NEXT CELL — find interacting medicines inside each statement

def find_drug_b(statement, drug_a):
    text = str(statement).lower()
    drug_a_norm = str(drug_a).lower().strip()

    found = []

    for drug in known_drugs:

        # skip the medicine itself
        if drug == drug_a_norm:
            continue

        # avoid very short ambiguous terms
        if len(drug) < 4:
            continue

        pattern = r"(?<!\w)" + re.escape(drug) + r"(?!\w)"

        if re.search(pattern, text, flags=re.I):
            found.append(drug)

    return list(dict.fromkeys(found))

In [95]:
# NEXT CELL — create drug A ↔ drug B records

pair_rows = []

for _, row in interaction_records.iterrows():

    for statement in row["interaction_statements"]:

        matched_drugs = find_drug_b(
            statement,
            row["drug_a"]
        )

        for drug_b in matched_drugs:

            pair_rows.append({
                "drug_a": row["drug_a"],
                "drug_a_atc": row["drug_a_atc"],

                "drug_b": drug_b,
                "drug_b_atc": drug_name_to_atc.get(drug_b),

                "effect_warning": statement,

                "product_name": row["product_name"],
                "npl_id": row["npl_id"],

                "source": "FASS",
                "source_url": row["source_url"]
            })


pairs_df = pd.DataFrame(pair_rows)

print("Extracted interaction records:", len(pairs_df))

pairs_df.head(20)

Extracted interaction records: 42620


,drug_a,drug_a_atc,drug_b,drug_b_atc,effect_warning,product_name,npl_id,source,source_url
0,Insulin glargin,A10AE04,fluoxetin,N06AB03,Substanser som kan öka den blodsockersänkande ...,ABASAGLAR,20130705000068,FASS,https://fass.se/health/product/20130705000068/...
1,Insulin glargin,A10AE04,salbutamol,R03AC02,Substanser som kan minska den blodsockersänkan...,ABASAGLAR,20130705000068,FASS,https://fass.se/health/product/20130705000068/...
2,Insulin glargin,A10AE04,terbutalin,R03AC03,Substanser som kan minska den blodsockersänkan...,ABASAGLAR,20130705000068,FASS,https://fass.se/health/product/20130705000068/...
3,Insulin glargin,A10AE04,somatropin,H01AC01,Substanser som kan minska den blodsockersänkan...,ABASAGLAR,20130705000068,FASS,https://fass.se/health/product/20130705000068/...
4,Insulin glargin,A10AE04,adrenalin,C01CA24,Substanser som kan minska den blodsockersänkan...,ABASAGLAR,20130705000068,FASS,https://fass.se/health/product/20130705000068/...
5,Insulin glargin,A10AE04,olanzapin,N05AH03,Substanser som kan minska den blodsockersänkan...,ABASAGLAR,20130705000068,FASS,https://fass.se/health/product/20130705000068/...
6,Insulin glargin,A10AE04,glukagon,H04AA01,Substanser som kan minska den blodsockersänkan...,ABASAGLAR,20130705000068,FASS,https://fass.se/health/product/20130705000068/...
7,Insulin glargin,A10AE04,klozapin,N05AH02,Substanser som kan minska den blodsockersänkan...,ABASAGLAR,20130705000068,FASS,https://fass.se/health/product/20130705000068/...
8,Insulin glargin,A10AE04,fluoxetin,N06AB03,Substanser som kan öka den blodsockersänkande ...,ABASAGLAR KwikPen,20130705000075,FASS,https://fass.se/health/product/20130705000075/...
9,Insulin glargin,A10AE04,salbutamol,R03AC02,Substanser som kan minska den blodsockersänkan...,ABASAGLAR KwikPen,20130705000075,FASS,https://fass.se/health/product/20130705000075/...


In [96]:
pairs_df[
    pairs_df["product_name"]
    .str.contains("Abboticin", case=False, na=False)
][
    [
        "drug_a",
        "drug_b",
        "drug_a_atc",
        "drug_b_atc",
        "effect_warning"
    ]
].head(30)

,drug_a,drug_b,drug_a_atc,drug_b_atc,effect_warning
16,Erytromycin,metylprednisolon,J01FA01,H02AB04,En dosreduktion kan vara nödvändig av andra lä...
17,Erytromycin,ketokonazol,J01FA01,D01AC08,En dosreduktion kan vara nödvändig av andra lä...
18,Erytromycin,ciklosporin,J01FA01,S01XA18,En dosreduktion kan vara nödvändig av andra lä...
19,Erytromycin,itrakonazol,J01FA01,J02AC02,En dosreduktion kan vara nödvändig av andra lä...
20,Erytromycin,takrolimus,J01FA01,D11AH01,En dosreduktion kan vara nödvändig av andra lä...
21,Erytromycin,flukonazol,J01FA01,J02AC01,En dosreduktion kan vara nödvändig av andra lä...
22,Erytromycin,midazolam,J01FA01,N05CD08,En dosreduktion kan vara nödvändig av andra lä...
23,Erytromycin,omeprazol,J01FA01,A02BC01,En dosreduktion kan vara nödvändig av andra lä...
24,Erytromycin,kolkicin,J01FA01,M04AC01,En dosreduktion kan vara nödvändig av andra lä...
25,Erytromycin,sildenafil,J01FA01,G04BE03,Sildenafil Data tyder på att erytromycin hämma...


In [97]:
# NEXT CELL
# Canonical pair key + merge duplicate interaction statements

def make_pair_key(row):
    a = str(row["drug_a_atc"]).strip().upper()
    b = str(row["drug_b_atc"]).strip().upper()
    return "||".join(sorted([a, b]))

pairs_df["pair_key"] = pairs_df.apply(make_pair_key, axis=1)

# Remove exact duplicate warning text first
pairs_df["warning_key"] = (
    pairs_df["effect_warning"]
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

pairs_unique = pairs_df.drop_duplicates(
    subset=["pair_key", "warning_key"]
).copy()

print("Before:", len(pairs_df))
print("After exact duplicate removal:", len(pairs_unique))

Before: 42620
After exact duplicate removal: 10713


In [98]:
# Merge all unique warnings for the same drug pair

interaction_pairs = (
    pairs_unique
    .groupby(
        [
            "pair_key",
            "drug_a",
            "drug_a_atc",
            "drug_b",
            "drug_b_atc"
        ],
        as_index=False
    )
    .agg({
        "effect_warning": lambda x: " | ".join(dict.fromkeys(x)),
        "source": "first",
        "source_url": lambda x: " | ".join(dict.fromkeys(x.astype(str))),
        "product_name": lambda x: " | ".join(dict.fromkeys(x.astype(str)))
    })
)

print("Final unique interaction pairs:", len(interaction_pairs))

interaction_pairs.head(30)

Final unique interaction pairs: 2821


,pair_key,drug_a,drug_a_atc,drug_b,drug_b_atc,effect_warning,source,source_url,product_name
0,A02AD01||A06AG07,Kalciumkarbonat + magnesiumhydroxid,A02AD01,sorbitol,A06AG07,Innehåll av sorbitol i läkemedel för oralt bru...,FASS,https://fass.se/health/product/19910308000030/...,Novalucol
1,A02AD01||A12AA04,Kalciumkarbonat + magnesiumhydroxid,A02AD01,kalcium,A12AA04,"Estramustin Kalcium, magnesium och aluminium b...",FASS,https://fass.se/health/product/19910308000030/...,Novalucol
2,A02AD01||B03AB10,Kalciumkarbonat + magnesiumhydroxid,A02AD01,järn,B03AB10,"Tvåvärt järn, perorala järnpreparat: Magnesium...",FASS,https://fass.se/health/product/20190619000059/...,Abecol | Novalucol | Refluxid
3,A02AD01||B03AB10,Kalciumkarbonat + magnesiumkarbonat,A02AD01,järn,B03AB10,"Tvåvärt järn, perorala järnpreparat: Magnesium...",FASS,https://fass.se/health/product/19760225000039/...,Rennie
4,A02AD01||C01AA05,Kalciumkarbonat + magnesiumhydroxid,A02AD01,digoxin,C01AA05,Absorptionen av följande peroralt administrera...,FASS,https://fass.se/health/product/19910308000030/...,Novalucol
5,A02AD01||D01AC08,Kalciumkarbonat + magnesiumhydroxid,A02AD01,ketokonazol,D01AC08,Ketokonazol : Upplösningen i magsäcken försämr...,FASS,https://fass.se/health/product/20190619000059/...,Abecol | Novalucol
6,A02AD01||D01AC08,Kalciumkarbonat + magnesiumkarbonat,A02AD01,ketokonazol,D01AC08,Ketokonazol: Upplösningen i magsäcken försämra...,FASS,https://fass.se/health/product/19760225000039/...,Rennie
7,A02AD01||G04BX01,Kalciumkarbonat + magnesiumhydroxid,A02AD01,magnesium,G04BX01,Penicillamin : Kan chelatbindas till magnesium...,FASS,https://fass.se/health/product/20190619000059/...,Abecol | Novalucol
8,A02AD01||G04BX01,Kalciumkarbonat + magnesiumkarbonat,A02AD01,magnesium,G04BX01,Penicillamin: Kan chelatbindas till magnesium ...,FASS,https://fass.se/health/product/19760225000039/...,Rennie
9,A02AD01||J01AA02,Kalciumkarbonat + magnesiumhydroxid,A02AD01,doxycyklin,J01AA02,Det har nyligen visats att peroralt aluminiumh...,FASS,https://fass.se/health/product/19910308000030/...,Novalucol


In [99]:
interaction_pairs[
    interaction_pairs["drug_a"]
    .str.contains("Erytromycin", case=False, na=False)
][
    [
        "drug_a",
        "drug_b",
        "drug_a_atc",
        "drug_b_atc",
        "effect_warning"
    ]
].head(30)

,drug_a,drug_b,drug_a_atc,drug_b_atc,effect_warning
37,Erytromycin,omeprazol,J01FA01,A02BC01,En dosreduktion kan vara nödvändig av andra lä...
546,Erytromycin,warfarin,J01FA01,B01AA03,Antikoagulantia Det har rapporterats om ökade ...
769,Erytromycin,rivaroxaban,J01FA01,B01AF01,Antikoagulantia Det har rapporterats om ökade ...
884,Erytromycin,digoxin,J01FA01,C01AA05,Digoxin Samtidig användning av erytromycin och...
1289,Erytromycin,verapamil,J01FA01,C08DA01,"Hypotension, bradyarytmier och laktoacidos har..."
1401,Erytromycin,simvastatin,J01FA01,C10AA01,"Rhabdomyolos, som associerats med förhöjda pla..."
1436,Erytromycin,atorvastatin,J01FA01,C10AA05,Abboticin ska inte användas samtidigt som simv...
1519,Erytromycin,ketokonazol,J01FA01,D01AC08,En dosreduktion kan vara nödvändig av andra lä...
1612,Erytromycin,takrolimus,J01FA01,D11AH01,En dosreduktion kan vara nödvändig av andra lä...
1636,Erytromycin,klindamycin,J01FA01,G01AA10,"Erytromycin motverkar effekten av klindamycin,..."


In [100]:
import re

INTERACTION_CUES = [
    r"\bsamtidig",
    r"\bkombination",
    r"\binteraktion",
    r"\bkontraindicer",
    r"\bska inte användas",
    r"\bbör undvikas",
    r"\bförsiktighet",
    r"\bhämmar",
    r"\binducerar",
    r"\bökar",
    r"\bminskar",
    r"\bförhöjd",
    r"\bplasmakoncentration",
    r"\bdosreduktion",
    r"\bdosjustering",
    r"\btoxicitet",
    r"\brisk",
]

def has_interaction_cue(text):
    text = str(text).lower()

    return any(
        re.search(pattern, text)
        for pattern in INTERACTION_CUES
    )


interaction_pairs["has_interaction_cue"] = (
    interaction_pairs["effect_warning"]
    .apply(has_interaction_cue)
)

filtered_pairs = interaction_pairs[
    interaction_pairs["has_interaction_cue"]
].copy()

print("Before:", len(interaction_pairs))
print("After cue filtering:", len(filtered_pairs))

Before: 2821
After cue filtering: 2308


In [101]:
filtered_pairs[
    filtered_pairs["drug_a"]
    .str.contains("Erytromycin", case=False, na=False)
][
    [
        "drug_a",
        "drug_b",
        "drug_a_atc",
        "drug_b_atc",
        "effect_warning"
    ]
]

,drug_a,drug_b,drug_a_atc,drug_b_atc,effect_warning
37,Erytromycin,omeprazol,J01FA01,A02BC01,En dosreduktion kan vara nödvändig av andra lä...
546,Erytromycin,warfarin,J01FA01,B01AA03,Antikoagulantia Det har rapporterats om ökade ...
769,Erytromycin,rivaroxaban,J01FA01,B01AF01,Antikoagulantia Det har rapporterats om ökade ...
884,Erytromycin,digoxin,J01FA01,C01AA05,Digoxin Samtidig användning av erytromycin och...
1289,Erytromycin,verapamil,J01FA01,C08DA01,"Hypotension, bradyarytmier och laktoacidos har..."
1401,Erytromycin,simvastatin,J01FA01,C10AA01,"Rhabdomyolos, som associerats med förhöjda pla..."
1436,Erytromycin,atorvastatin,J01FA01,C10AA05,Abboticin ska inte användas samtidigt som simv...
1519,Erytromycin,ketokonazol,J01FA01,D01AC08,En dosreduktion kan vara nödvändig av andra lä...
1612,Erytromycin,takrolimus,J01FA01,D11AH01,En dosreduktion kan vara nödvändig av andra lä...
1827,Erytromycin,sildenafil,J01FA01,G04BE03,Sildenafil Data tyder på att erytromycin hämma...
